In [1]:
import pathlib
import json
import shutil
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers, callbacks

# ── Constants ────────────────────────────────────────────────
RANDOM_SEED  = 42
SEQUENCE_LEN = 60
NUM_FEATURES = 126
BATCH_SIZE   = 32
EPOCHS       = 100

DATA_DIR  = pathlib.Path("../data/hand_v3")
MODEL_DIR = pathlib.Path("../saved_models/v3")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"TensorFlow : {tf.__version__}")

TensorFlow : 2.21.0


In [2]:
X_train = np.load(DATA_DIR / "X_train_aug.npy")
y_train = np.load(DATA_DIR / "y_train_aug.npy")

X_val   = np.load(DATA_DIR / "X_val_hand.npy")
y_val   = np.load(DATA_DIR / "y_val_hand.npy")

X_test  = np.load(DATA_DIR / "X_test_hand.npy")
y_test  = np.load(DATA_DIR / "y_test_hand.npy")

NUM_CLASSES = y_train.shape[1]

print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")
print(f"Classes : {NUM_CLASSES}")

X_train : (23365, 60, 126)
X_val   : (388, 60, 126)
X_test  : (778, 60, 126)
Classes : 312


In [3]:
def topk_accuracy(model, X, y, k):
    preds = model.predict(X, verbose=0)
    true_labels = np.argmax(y, axis=1)
    topk_preds = np.argsort(preds, axis=1)[:, -k:]
    correct = sum(t in p for t, p in zip(true_labels, topk_preds))
    return correct / len(true_labels)


def eval_model(model, X, y, name):
    _, top1 = model.evaluate(X, y, verbose=0)
    top3 = topk_accuracy(model, X, y, k=3)
    top5 = topk_accuracy(model, X, y, k=5)
    print(f"  {name:<25}  Top-1: {top1*100:.2f}%   Top-3: {top3*100:.2f}%   Top-5: {top5*100:.2f}%")
    return top1, top3, top5

In [4]:
def build_gru(sequence_len, num_features, num_classes):
    """
    Lightweight unidirectional GRU companion.
    Input (60, 126)
    → GRU(128, return_sequences=True) + BatchNorm + Dropout(0.3)
    → GRU(64) + BatchNorm + Dropout(0.3)
    → Dense(128, relu) + Dropout(0.3)
    → Dense(num_classes, softmax)
    """
    inp = keras.Input(shape=(sequence_len, num_features))
    x = layers.GRU(128, return_sequences=True)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.GRU(64)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs=inp, outputs=out, name="gru_v2")


gru_model = build_gru(SEQUENCE_LEN, NUM_FEATURES, NUM_CLASSES)
gru_model.summary()

Model: "gru_v2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 60, 126)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 60, 128)        │        98,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 60, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 312)            │        40,248 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 184,888 (722.22 KB)

 Trainable params: 184,504 (720.72 KB)

 Non-trainable params: 384 (1.50 KB)

In [5]:
gru_checkpoint = str(MODEL_DIR / "gru_hand_v2_best.keras")

gru_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

gru_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[
        callbacks.EarlyStopping(monitor="val_accuracy", patience=15, restore_best_weights=True),
        callbacks.ModelCheckpoint(gru_checkpoint, monitor="val_accuracy", save_best_only=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6, min_lr=1e-6),
    ],
    verbose=1,
    )
    
gru_model = keras.models.load_model(gru_checkpoint)
print(f"GRU model saved: {gru_checkpoint}")

Epoch 1/100
731/731 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - accuracy: 0.0501 - loss: 4.8863 - val_accuracy: 0.1263 - val_loss: 4.0978 - learning_rate: 0.0010
Epoch 2/100
731/731 ━━━━━━━━━━━━━━━━━━━━ 16s 22ms/step - accuracy: 0.1771 - loss: 3.4363 - val_accuracy: 0.2216 - val_loss: 3.2327 - learning_rate: 0.0010
Epoch 3/100
731/731 ━━━━━━━━━━━━━━━━━━━━ 19s 26ms/step - accuracy: 0.3173 - loss: 2.5693 - val_accuracy: 0.3196 - val_loss: 2.8354 - learning_rate: 0.0010
Epoch 4/100
731/731 ━━━━━━━━━━━━━━━━━━━━ 19s 26ms/step - accuracy: 0.4225 - loss: 2.0523 - val_accuracy: 0.3840 - val_loss: 2.6771 - learning_rate: 0.0010
Epoch 5/100
731/731 ━━━━━━━━━━━━━━━━━━━━ 19s 26ms/step - accuracy: 0.5035 - loss: 1.6878 - val_accuracy: 0.4201 - val_loss: 2.5807 - learning_rate: 0.0010
Epoch 6/100
731/731 ━━━━━━━━━━━━━━━━━━━━ 17s 23ms/step - accuracy: 0.5742 - loss: 1.4248 - val_accuracy: 0.4201 - val_loss: 2.7188 - learning_rate: 0.0010
Epoch 7/100
731/731 ━━━━━━━━━━━━━━━━━━━━ 16s 22ms/step - accuracy: 0.6

In [6]:
def build_cnn(sequence_len, num_features, num_classes):
    """
    1D-CNN companion model.
    Input (60, 126)
    → Conv1D(64, 3) + BatchNorm + MaxPool(2) + Dropout(0.2)
    → Conv1D(128, 3) + BatchNorm + MaxPool(2) + Dropout(0.2)
    → GlobalAveragePooling1D
    → Dense(128, relu) + Dropout(0.3)
    → Dense(num_classes, softmax)
    """
    inp = keras.Input(shape=(sequence_len, num_features))
    x = layers.Conv1D(64, kernel_size=3, padding="same", activation="relu")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Conv1D(128, kernel_size=3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs=inp, outputs=out, name="cnn_v2")


cnn_model = build_cnn(SEQUENCE_LEN, NUM_FEATURES, NUM_CLASSES)
cnn_model.summary()

Model: "cnn_v2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 60, 126)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 60, 64)         │        24,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 60, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 30, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 30, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 15, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 15, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 312)            │        40,248 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 106,488 (415.97 KB)

 Trainable params: 106,104 (414.47 KB)

 Non-trainable params: 384 (1.50 KB)

In [7]:
cnn_checkpoint = str(MODEL_DIR / "cnn_hand_v2_best.keras")

cnn_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

cnn_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[
        callbacks.EarlyStopping(monitor="val_accuracy", patience=15, restore_best_weights=True),
        callbacks.ModelCheckpoint(cnn_checkpoint, monitor="val_accuracy", save_best_only=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6, min_lr=1e-6),
    ],
    verbose=1,
)

cnn_model = keras.models.load_model(cnn_checkpoint)
print(f"CNN model saved: {cnn_checkpoint}")

Epoch 1/100
731/731 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.1788 - loss: 3.7995 - val_accuracy: 0.3015 - val_loss: 2.9957 - learning_rate: 0.0010
Epoch 2/100
731/731 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.4054 - loss: 2.2246 - val_accuracy: 0.3789 - val_loss: 2.5676 - learning_rate: 0.0010
Epoch 3/100
731/731 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.5264 - loss: 1.6833 - val_accuracy: 0.4201 - val_loss: 2.3855 - learning_rate: 0.0010
Epoch 4/100
731/731 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6006 - loss: 1.3674 - val_accuracy: 0.4742 - val_loss: 2.4396 - learning_rate: 0.0010
Epoch 5/100
731/731 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6520 - loss: 1.1606 - val_accuracy: 0.4562 - val_loss: 2.4349 - learning_rate: 0.0010
Epoch 6/100
731/731 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6893 - loss: 1.0207 - val_accuracy: 0.5052 - val_loss: 2.4433 - learning_rate: 0.0010
Epoch 7/100
731/731 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7173 - loss: 0.

In [8]:
bigru_checkpoint = str(MODEL_DIR / "bigru_attn_v2_best.keras")
bigru_model = keras.models.load_model(bigru_checkpoint)
print(f"BiGRU+Attention model loaded from: {bigru_checkpoint}")

BiGRU+Attention model loaded from: ../saved_models/v3/bigru_attn_v2_best.keras


In [9]:
print("\nIndividual Model Results on Test Set:")
print("─" * 65)

bigru_scores = eval_model(bigru_model, X_test, y_test, "BiGRU + Attention")
gru_scores   = eval_model(gru_model,   X_test, y_test, "GRU")
cnn_scores   = eval_model(cnn_model,   X_test, y_test, "CNN")


Individual Model Results on Test Set:
─────────────────────────────────────────────────────────────────
  BiGRU + Attention          Top-1: 61.57%   Top-3: 78.66%   Top-5: 84.06%
  GRU                        Top-1: 55.53%   Top-3: 71.34%   Top-5: 78.28%
  CNN                        Top-1: 60.28%   Top-3: 77.76%   Top-5: 83.16%


In [10]:
# Predict probabilities from all 3 models
p_bigru = bigru_model.predict(X_test, verbose=0)
p_gru   = gru_model.predict(X_test,   verbose=0)
p_cnn   = cnn_model.predict(X_test,   verbose=0)

# Equal-weight average
p_ensemble = (p_bigru + p_gru + p_cnn) / 3.0

# Evaluate ensemble
true_labels = np.argmax(y_test, axis=1)

def topk_from_probs(probs, true_labels, k):
    topk_preds = np.argsort(probs, axis=1)[:, -k:]
    correct = sum(t in p for t, p in zip(true_labels, topk_preds))
    return correct / len(true_labels)

ens_top1 = topk_from_probs(p_ensemble, true_labels, k=1)
ens_top3 = topk_from_probs(p_ensemble, true_labels, k=3)
ens_top5 = topk_from_probs(p_ensemble, true_labels, k=5)

print(f"\n  {'Ensemble (avg 3 models)':<25}  Top-1: {ens_top1*100:.2f}%   Top-3: {ens_top3*100:.2f}%   Top-5: {ens_top5*100:.2f}%")


  Ensemble (avg 3 models)    Top-1: 66.58%   Top-3: 79.82%   Top-5: 84.70%


In [11]:
print("\n")
print(f"{'Model':<28} {'Top-1':>8} {'Top-3':>8} {'Top-5':>8}")
print("─" * 58)
print(f"{'BiGRU + Attention':<28} {bigru_scores[0]*100:>7.2f}% {bigru_scores[1]*100:>7.2f}% {bigru_scores[2]*100:>7.2f}%")
print(f"{'GRU':<28} {gru_scores[0]*100:>7.2f}% {gru_scores[1]*100:>7.2f}% {gru_scores[2]*100:>7.2f}%")
print(f"{'CNN':<28} {cnn_scores[0]*100:>7.2f}% {cnn_scores[1]*100:>7.2f}% {cnn_scores[2]*100:>7.2f}%")
print(f"{'Ensemble':<28} {ens_top1*100:>7.2f}% {ens_top3*100:>7.2f}% {ens_top5*100:>7.2f}%")
print("─" * 58)



Model                           Top-1    Top-3    Top-5
──────────────────────────────────────────────────────────
BiGRU + Attention              61.57%   78.66%   84.06%
GRU                            55.53%   71.34%   78.28%
CNN                            60.28%   77.76%   83.16%
Ensemble                       66.58%   79.82%   84.70%
──────────────────────────────────────────────────────────


In [12]:
# Find best single model by test Top-1
model_results = {
    "bigru": (bigru_scores[0], bigru_checkpoint),
    "gru":   (gru_scores[0],   gru_checkpoint),
    "cnn":   (cnn_scores[0],   cnn_checkpoint),
}

best_name, (best_top1, best_path) = max(model_results.items(), key=lambda x: x[1][0])
candidate_path = MODEL_DIR / "mudralearn_v2_candidate.keras"

shutil.copy(best_path, str(candidate_path))
print(f"Best single model : {best_name} — {best_top1*100:.2f}% Top-1")
print(f"Promoted to       : {candidate_path}")
print()
print("NOTE: Do NOT overwrite mudralearn_model.keras yet.")
print("      That happens only after notebook 08 (evaluation gate) PASSES.")

Best single model : bigru — 61.57% Top-1
Promoted to       : ../saved_models/v3/mudralearn_v2_candidate.keras

NOTE: Do NOT overwrite mudralearn_model.keras yet.
      That happens only after notebook 08 (evaluation gate) PASSES.


In [13]:
print("=" * 50)
print("FINDINGS SUMMARY")
print("=" * 50)
print(f"Best single model : {best_name} at {best_top1*100:.2f}% Top-1")
print(f"Ensemble Top-1    : {ens_top1*100:.2f}%")
print(f"Ensemble Top-3    : {ens_top3*100:.2f}%")
print(f"Ensemble Top-5    : {ens_top5*100:.2f}%")
print(f"Candidate saved   : {candidate_path}")
print("=" * 50)

FINDINGS SUMMARY
Best single model : bigru at 61.57% Top-1
Ensemble Top-1    : 66.58%
Ensemble Top-3    : 79.82%
Ensemble Top-5    : 84.70%
Candidate saved   : ../saved_models/v3/mudralearn_v2_candidate.keras
